# Home Assignment 3: Bayes and Naive Bayes
Roll No: 230122018

In [1]:
import numpy as np
from scipy.stats import beta, norm
np.set_printoptions(precision=5, suppress=True)

## Q1. Conditional probability
Rows = [perovskite, not perovskite], columns = [oxide, not oxide]. Find the joint probability and both conditional probabilities.

In [2]:
table = np.array([[4, 2], [3, 6]])
both = table[0, 0]

p_joint = both / table.sum()
p_p_given_o = both / table[:, 0].sum()
p_o_given_p = both / table[0, :].sum()

print(p_joint, p_p_given_o, p_o_given_p)
assert np.allclose([p_joint, p_p_given_o, p_o_given_p], [4/15, 4/7, 4/6])

0.26666666666666666 0.5714285714285714 0.6666666666666666


**Why do the two conditional probabilities differ?**

The numerator is 4 in both, but I divided by 7 oxides in one and 6 perovskites in the other, so the answers come out different.

## Q2. One positive screening result
Prior of stable = 0.10. Filter flags 80% of stable and 20% of unstable materials. Find P(flag) and P(stable | flag).

In [3]:
p_s, tpr, fpr = 0.10, 0.80, 0.20

p_flag = tpr * p_s + fpr * (1 - p_s)
posterior = tpr * p_s / p_flag

print(p_flag, posterior)
assert np.isclose(p_flag, 0.26) and np.isclose(posterior, 0.08 / 0.26)

0.26 0.30769230769230776


**Why is the answer different from 0.80?**

0.80 is P(flag | stable), but I need P(stable | flag). Unstable materials are much more common, so their false flags (0.18) are more than the true flags (0.08), which brings the answer down to about 0.31.

## Q3. Second independent filter
Write a Bayes update function and apply it twice starting from 0.10.

In [4]:
def bayes_positive(p_h, tpr, fpr):
    return tpr * p_h / (tpr * p_h + fpr * (1 - p_h))

first = bayes_positive(0.10, 0.80, 0.20)
second = bayes_positive(first, 0.80, 0.20)

print(first, second)
assert np.isclose(first, 4/13) and np.isclose(second, 0.64)

0.30769230769230776 0.64


**Would the same deterministic filter applied twice justify this?**

No. The same filter on the same sample will just give the same result again, so it is not new evidence. I can only do the second update if the second test is actually independent.

## Q4. Beta update
Prior Beta(3, 7), then 4 stable out of 12 trials. Find the posterior parameters and mean.

In [5]:
a0, b0, k, n = 3, 7, 4, 12

a_post = a0 + k
b_post = b0 + (n - k)
post_mean = beta(a_post, b_post).mean()

print(a_post, b_post, post_mean)
assert (a_post, b_post) == (7, 15) and np.isclose(post_mean, 7/22)

7 15 0.3181818181818182


**Compare the posterior mean with the prior mean and observed fraction.**

Prior mean is 0.30, observed fraction is 0.333 and my posterior mean is 0.318. It lies in between, closer to the data because 12 trials is a bit more than the prior's weight of 10.

## Q5. Discrete classification
Classify a non-oxide with CN = 6 and density below 5 g/cm³. Rows = [Insulator, Metal], columns = [oxide, CN=6, low density].

In [6]:
prior = np.array([2/3, 1/3])
p = np.array([[0.8, 0.5, 0.6], [0.3, 0.3, 0.4]])
present = np.array([False, True, True])

lik = np.where(present, p, 1 - p)
scores = prior * lik.prod(axis=1)
posterior = scores / scores.sum()

print(scores, posterior)
assert np.allclose(scores, [0.04, 0.028])

[0.04  0.028] [0.58824 0.41176]


**Which class wins, and why is it less decisive than for an oxide?**

Insulator wins (0.59 vs 0.41). Being a non-oxide actually supports Metal (0.7 vs 0.2), so it pulls against the other features. If it were an oxide, all features would favour Insulator and the result would be around 0.93.

## Q6. Laplace smoothing
0 of 20 metals and 8 of 40 insulators have the feature. Find smoothed P(yes | class).

In [7]:
p_yes_metal = (0 + 1) / (20 + 2)
p_yes_insulator = (8 + 1) / (40 + 2)

print(p_yes_metal, p_yes_insulator)
assert np.allclose([p_yes_metal, p_yes_insulator], [1/22, 9/42])

0.045454545454545456 0.21428571428571427


**Why doesn't a zero count mean impossible?**

20 metals is a small sample, so the feature may just be rare and not show up. If I kept it as 0, the whole Metal score would become 0, so smoothing keeps it small but not zero.

## Q7. Gaussian density
Evaluate x = 0.7 with mean 0.6 Å and std 0.2 Å, and find the z-score.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, e^{-\frac{(x-\mu)^2}{2\sigma^2}}, \qquad z = \frac{x-\mu}{\sigma}$$

In [8]:
def gaussian_pdf(x, mu, sigma):
    return np.exp(-(x - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))

z = (0.7 - 0.6) / 0.2
density = gaussian_pdf(0.7, 0.6, 0.2)

print(z, density, norm.pdf(0.7, 0.6, 0.2))
assert np.isclose(z, 0.5) and np.isclose(density, 1.7603266338)

0.4999999999999999 1.7603266338214976 1.7603266338214976


**Why can the density be more than one?**

Density is not a probability, it is the height of the curve. Only the area has to be 1, and since σ = 0.2 is small the curve is narrow, so it has to be tall.

## Q8. Gaussian Naive Bayes
Classify x = [5.0, 0.7, 0.6] (density, atomic size, electronegativity difference). Find the 2×3 density array, scores and probabilities.

In [9]:
x = np.array([5.0, 0.7, 0.6])
mu = np.array([[4, 1.3, 2.1], [4, 0.6, 0.4]])
sd = np.array([[2, 0.2, 0.4], [2, 0.2, 0.3]])
prior = np.array([2/3, 1/3])

heights = gaussian_pdf(x, mu, sd)
scores = prior * heights.prod(axis=1)
posterior = scores / scores.sum()

print(heights)
print(scores)
print(posterior)
assert heights.shape == (2, 3) and np.isclose(posterior.sum(), 1)

[[0.17603 0.02216 0.00088]
 [0.17603 1.76033 1.06483]]
[0.      0.10999]
[0.00002 0.99998]


**Why doesn't density help, and is this probability experimentally proven?**

Density has the same mean and std for both classes, so it gives the same value and cancels out. The probability only comes from the numbers I assumed in the model, so it is not proven by experiment and would need real data to check.